# Análise Exploratória de Dados - Sistema de Controle de Subvenções (SCS/ANEEL)

**Instituto Federal de Educação, Ciência e Tecnologia de Goiás**

**Mestrado Profissional em Tecnologia, Gestão e Sustentabilidade**

**Métodos de Previsão**

**Prof. Dr. Raphael Gomes**

Este notebook contém funções para análise exploratória de dados.

## Objetivo
Este notebook explora o dataset do **Sistema de Controle de Subvenções e Programas Sociais** da ANEEL, aplicando conceitos de **Estatística e Análise Exploratória de Dados** vistos na Aula 02 do curso de Métodos de Previsão.

## Sobre o Dataset
O dataset contém informações mensais sobre os descontos na tarifa de energia elétrica concedidos aos consumidores da subclasse residencial baixa renda, incluindo:
- Quantidade de consumidores por categoria (baixa renda, indígena, quilombola, BPC, multifamiliar)
- Energia faturada (MWh) por categoria
- Faturamento por categoria
- Valores de Diferença Mensal de Receita (DMR)
- Faixas de consumo (1 a 5)

**Fonte:** ANEEL - Agência Nacional de Energia Elétrica  
**URL:** https://dadosabertos.aneel.gov.br/dataset/scs-sistema-de-controle-de-subvencoes-e-programas-sociais

## 1. Configuração Inicial e Importação de Bibliotecas

Primeiro, importamos as bibliotecas necessárias para análise de dados e visualização.

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Configurações de visualização
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configuração para exibir todas as colunas do DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('✓ Bibliotecas importadas com sucesso!')

## 2. Carregamento e Exploração Inicial dos Dados

### 2.1 Leitura do Dataset

Como o dataset não está disponível localmente, vamos **baixá-lo diretamente do portal de dados abertos da ANEEL**.

In [ ]:
# URL do dataset no portal de dados abertos da ANEEL
url = 'https://dadosabertos.aneel.gov.br/dataset/942de3e1-0b52-4e41-a6c1-eff9f3b7c7d6/resource/87764789-84c3-4592-a845-cb2b317f6142/download/sistema-controle-subvencoes-programas-sociais.csv'

# Leitura do arquivo CSV
# O separador é ';' e o encoding é 'latin-1' conforme padrão dos dados abertos brasileiros
df = pd.read_csv(url, sep=';', encoding='latin-1', decimal=',')

print('✓ Dataset carregado com sucesso!')
print(f'\nDimensões do dataset: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

### 2.2 Visão Geral do Dataset

Conforme visto na aula, a primeira etapa da análise exploratória é **descrever o conjunto de dados**. Vamos examinar:
- As primeiras linhas
- Tipos de dados
- Informações gerais sobre as colunas

In [ ]:
# Exibir as primeiras 5 linhas do dataset
print('=== PRIMEIRAS 5 LINHAS DO DATASET ===')
print(df.head())

In [ ]:
# Informações sobre o dataset: tipos de dados, memória, valores nulos
print('\n=== INFORMAÇÕES SOBRE O DATASET ===')
print(df.info())

In [ ]:
# Nome das colunas
print('\n=== COLUNAS DO DATASET ===')
for i, col in enumerate(df.columns, 1):
    print(f'{i:2d}. {col}')

## 3. Limpeza e Preparação dos Dados

Antes de realizar análises estatísticas, é fundamental limpar e preparar os dados.

In [ ]:
# Verificar valores ausentes (missing values)
print('=== VALORES AUSENTES POR COLUNA ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Valores_Nulos': missing,
    'Percentual (%)': missing_pct
})
# Filtrar apenas colunas com valores ausentes
missing_df = missing_df[missing_df['Valores_Nulos'] > 0].sort_values('Valores_Nulos', ascending=False)

if len(missing_df) > 0:
    print(missing_df)
else:
    print('✓ Não há valores ausentes no dataset!')

In [ ]:
# Criar cópia do dataset para trabalhar
df_clean = df.copy()

# Converter colunas de data para formato datetime
if 'DatGeracaoConjuntoDados' in df_clean.columns:
    df_clean['DatGeracaoConjuntoDados'] = pd.to_datetime(df_clean['DatGeracaoConjuntoDados'], errors='coerce')

if 'DatRegistro' in df_clean.columns:
    df_clean['DatRegistro'] = pd.to_datetime(df_clean['DatRegistro'], errors='coerce')

# Converter AnmMesAnoCompetencia para datetime (formato YYYYMM)
if 'AnmMesAnoCompetencia' in df_clean.columns:
    df_clean['AnmMesAnoCompetencia'] = pd.to_datetime(df_clean['AnmMesAnoCompetencia'], format='%Y%m', errors='coerce')

print('✓ Dados limpos e preparados!')
print(f'\nDataset final: {df_clean.shape[0]:,} linhas x {df_clean.shape[1]} colunas')

## 4. Estatísticas Descritivas - Tendências Centrais

Como visto na aula, as **medidas de tendência central** (média, mediana e moda) nos ajudam a entender onde os dados estão centralizados.

### 4.1 Estatísticas Gerais

A função `describe()` do pandas fornece um resumo estatístico das colunas numéricas:
- **count**: número de observações
- **mean**: média aritmética (x̄)
- **std**: desvio padrão (s)
- **min**: valor mínimo
- **25%, 50%, 75%**: quartis (Q1, Q2/mediana, Q3)
- **max**: valor máximo

In [ ]:
# Estatísticas descritivas das variáveis numéricas
print('=== ESTATÍSTICAS DESCRITIVAS ===')
print(df_clean.describe().round(2))

### 4.2 Análise de Consumidores por Categoria

Vamos calcular as **medidas de tendência central** para as diferentes categorias de consumidores.

In [ ]:
# Selecionar colunas de número de consumidores
cols_consumidores = [
    'NumConsBaixaRenda',
    'NumConsIndigena',
    'NumConsQuilombola',
    'NumConsBPC',
    'NumConsMultifamiliar'
]

# Verificar quais colunas existem no dataset
cols_consumidores = [col for col in cols_consumidores if col in df_clean.columns]

if len(cols_consumidores) > 0:
    # Calcular estatísticas de tendência central
    tendencias = pd.DataFrame({
        'Média': df_clean[cols_consumidores].mean(),
        'Mediana': df_clean[cols_consumidores].median(),
        'Moda': df_clean[cols_consumidores].mode().iloc[0],
        'Mínimo': df_clean[cols_consumidores].min(),
        'Máximo': df_clean[cols_consumidores].max()
    })

    print('\n=== TENDÊNCIAS CENTRAIS - NÚMERO DE CONSUMIDORES POR CATEGORIA ===')
    print(tendencias.round(2))

    # Insights sobre média vs mediana
    print('\n📊 INSIGHTS:')
    for col in cols_consumidores:
        media = df_clean[col].mean()
        mediana = df_clean[col].median()
        if media > mediana * 1.2:  # Se média é 20% maior que mediana
            print(f'• {col}: Distribuição assimétrica positiva (cauda à direita) - média > mediana')
        elif media < mediana * 0.8:  # Se média é 20% menor que mediana
            print(f'• {col}: Distribuição assimétrica negativa (cauda à esquerda) - média < mediana')
        else:
            print(f'• {col}: Distribuição aproximadamente simétrica')

## 5. Medidas de Dispersão

Como aprendemos na aula, as **medidas de dispersão** indicam como os dados estão espalhados:
- **Range (Intervalo)**: diferença entre máximo e mínimo
- **Variância (s²)**: média dos quadrados dos desvios em relação à média
- **Desvio Padrão (s)**: raiz quadrada da variância
- **IQR (Intervalo Interquartil)**: Q3 - Q1 (mais robusto a outliers)

In [ ]:
# Calcular medidas de dispersão
if len(cols_consumidores) > 0:
    dispersao = pd.DataFrame({
        'Range': df_clean[cols_consumidores].max() - df_clean[cols_consumidores].min(),
        'Variância': df_clean[cols_consumidores].var(),
        'Desvio_Padrão': df_clean[cols_consumidores].std(),
        'Q1 (25%)': df_clean[cols_consumidores].quantile(0.25),
        'Q3 (75%)': df_clean[cols_consumidores].quantile(0.75),
        'IQR': df_clean[cols_consumidores].quantile(0.75) - df_clean[cols_consumidores].quantile(0.25),
        'CV (%)': (df_clean[cols_consumidores].std() / df_clean[cols_consumidores].mean()) * 100  # Coeficiente de variação
    })

    print('\n=== MEDIDAS DE DISPERSÃO ===')
    print(dispersao.round(2))

    print('\n📊 INTERPRETAÇÃO:')
    print('• CV (Coeficiente de Variação) < 15%: Baixa dispersão')
    print('• CV entre 15% e 30%: Média dispersão')
    print('• CV > 30%: Alta dispersão')

## 6. Visualização de Distribuições - Histogramas

Como visto na aula, **histogramas** são fundamentais para visualizar a distribuição dos dados.

In [ ]:
# Criar histogramas para número de consumidores por categoria
if len(cols_consumidores) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for i, col in enumerate(cols_consumidores):
        # Remover valores zero para melhor visualização
        data = df_clean[col][df_clean[col] > 0]

        # Histograma
        axes[i].hist(data, bins=30, edgecolor='black', alpha=0.7)
        axes[i].set_xlabel(col.replace('NumCons', '').replace('BaixaRenda', 'Baixa Renda'))
        axes[i].set_ylabel('Frequência')
        axes[i].set_title(f'Distribuição - {col.replace("NumCons", "")}', fontweight='bold')
        axes[i].grid(True, alpha=0.3)

        # Adicionar linha da média
        mean_val = data.mean()
        axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Média: {mean_val:.0f}')
        axes[i].legend()

    # Remover eixo extra
    if len(cols_consumidores) < len(axes):
        fig.delaxes(axes[-1])

    plt.tight_layout()
    plt.savefig('histogramas_consumidores.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('✓ Histogramas criados com sucesso!')

## 7. Box Plots - Detecção de Outliers

**Box plots** são excelentes para:
- Visualizar quartis (Q1, Q2/mediana, Q3)
- Identificar outliers (valores atípicos)
- Comparar distribuições entre categorias

In [ ]:
# Box plots para identificar outliers
if len(cols_consumidores) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))

    # Preparar dados removendo zeros
    data_boxplot = [df_clean[col][df_clean[col] > 0] for col in cols_consumidores]
    labels_boxplot = [col.replace('NumCons', '').replace('BaixaRenda', 'Baixa Renda') for col in cols_consumidores]

    # Criar box plot
    bp = ax.boxplot(data_boxplot, labels=labels_boxplot, patch_artist=True, showmeans=True)

    # Colorir as caixas
    colors = plt.cm.Set3(np.linspace(0, 1, len(cols_consumidores)))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    ax.set_ylabel('Número de Consumidores', fontsize=12, fontweight='bold')
    ax.set_title('Box Plot - Número de Consumidores por Categoria\n(Detecção de Outliers)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('boxplot_consumidores.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('✓ Box plot criado com sucesso!')
    print('\n📊 INTERPRETAÇÃO DO BOX PLOT:')
    print('• Caixa: Representa o IQR (Q1 a Q3) - contém 50% dos dados centrais')
    print('• Linha dentro da caixa: Mediana (Q2)')
    print('• Triângulo verde: Média')
    print('• Bigodes: Estendem-se até 1.5 × IQR')
    print('• Pontos fora dos bigodes: Outliers (valores atípicos)')

## 8. Análise de Faixas de Consumo

O dataset possui uma variável categórica **IdcFaixa** que indica a faixa de consumo:
- **1**: até 30 kWh
- **2**: 31 até 50 kWh
- **3**: 51 até 100 kWh
- **4**: 101 até 220 kWh
- **5**: acima de 220 kWh

In [ ]:
# Análise de faixas de consumo
if 'IdcFaixa' in df_clean.columns:
    print('=== DISTRIBUIÇÃO POR FAIXA DE CONSUMO ===')

    faixas_count = df_clean['IdcFaixa'].value_counts().sort_index()
    faixas_pct = (faixas_count / len(df_clean) * 100).round(2)

    faixas_df = pd.DataFrame({
        'Faixa': ['1 (≤30 kWh)', '2 (31-50 kWh)', '3 (51-100 kWh)', '4 (101-220 kWh)', '5 (>220 kWh)'],
        'Quantidade': faixas_count.values,
        'Percentual (%)': faixas_pct.values
    })

    print(faixas_df)
    print(f'\n📊 Faixa mais comum (Moda): Faixa {df_clean["IdcFaixa"].mode()[0]}')

In [ ]:
# Gráfico de barras - distribuição por faixa
if 'IdcFaixa' in df_clean.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Gráfico de barras
    faixas_labels = ['Faixa 1\n(≤30 kWh)', 'Faixa 2\n(31-50 kWh)', 'Faixa 3\n(51-100 kWh)',
                     'Faixa 4\n(101-220 kWh)', 'Faixa 5\n(>220 kWh)']
    colors_faixas = plt.cm.viridis(np.linspace(0, 1, len(faixas_count)))

    ax1.bar(faixas_labels, faixas_count.values, color=colors_faixas, edgecolor='black')
    ax1.set_ylabel('Quantidade de Registros', fontsize=11, fontweight='bold')
    ax1.set_title('Distribuição por Faixa de Consumo', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')

    # Adicionar valores nas barras
    for i, v in enumerate(faixas_count.values):
        ax1.text(i, v + max(faixas_count.values)*0.01, f'{v:,}', ha='center', va='bottom', fontweight='bold')

    # Gráfico de pizza
    ax2.pie(faixas_count.values, labels=faixas_labels, autopct='%1.1f%%',
            colors=colors_faixas, startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
    ax2.set_title('Proporção por Faixa de Consumo', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig('distribuicao_faixas.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('✓ Gráficos de distribuição por faixa criados!')

## 9. Análise de Correlação

Como visto na aula, a **correlação** mede a relação entre duas variáveis:
- **Covariância**: mede como duas variáveis variam juntas
- **Correlação de Pearson (r)**: normaliza a covariância pelos desvios padrões
  - r = 1: correlação positiva perfeita
  - r = 0: sem correlação linear
  - r = -1: correlação negativa perfeita

In [ ]:
# Selecionar variáveis numéricas relevantes para análise de correlação
cols_energia = [
    'MdaMWhBaixaRenda',
    'MdaMWhIndigena',
    'MdaMWhQuilombola',
    'MdaMWhBPC',
    'MdaMWhMultifamiliar'
]

cols_faturamento = [
    'VlrFatRealBaixaRenda',
    'VlrFatRealIndigena',
    'VlrFatRealQuilombola',
    'VlrFatRealBPC',
    'VlrMWhMultifamiliar'
]

# Verificar quais colunas existem
cols_energia = [col for col in cols_energia if col in df_clean.columns]
cols_faturamento = [col for col in cols_faturamento if col in df_clean.columns]
cols_correlacao = cols_consumidores + cols_energia + cols_faturamento

# Calcular matriz de correlação
if len(cols_correlacao) > 0:
    correlacao = df_clean[cols_correlacao].corr()

    print('=== MATRIZ DE CORRELAÇÃO (Primeiras 5x5) ===')
    print(correlacao.iloc[:5, :5].round(2))

In [ ]:
# Heatmap da matriz de correlação
if len(cols_correlacao) > 0:
    fig, ax = plt.subplots(figsize=(14, 12))

    # Criar heatmap
    sns.heatmap(correlacao, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={'label': 'Correlação de Pearson'},
                vmin=-1, vmax=1, ax=ax)

    ax.set_title('Matriz de Correlação - Heatmap\n(Correlação de Pearson)',
                 fontsize=14, fontweight='bold', pad=20)

    # Rotacionar labels
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)

    plt.tight_layout()
    plt.savefig('matriz_correlacao.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('✓ Heatmap de correlação criado!')
    print('\n📊 INTERPRETAÇÃO DA CORRELAÇÃO:')
    print('• |r| < 0.3: Correlação fraca')
    print('• 0.3 ≤ |r| < 0.7: Correlação moderada')
    print('• |r| ≥ 0.7: Correlação forte')
    print('• r > 0: Correlação positiva (crescem juntas)')
    print('• r < 0: Correlação negativa (uma cresce, outra diminui)')

### 9.1 Scatter Plots - Relação entre Variáveis

Vamos visualizar a **relação entre número de consumidores e energia faturada**.

In [ ]:
# Scatter plots para visualizar correlações
if 'NumConsBaixaRenda' in df_clean.columns and 'MdaMWhBaixaRenda' in df_clean.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    pares = [
        ('NumConsBaixaRenda', 'MdaMWhBaixaRenda', 'Baixa Renda'),
        ('NumConsIndigena', 'MdaMWhIndigena', 'Indígena'),
        ('NumConsQuilombola', 'MdaMWhQuilombola', 'Quilombola'),
        ('NumConsBPC', 'MdaMWhBPC', 'BPC')
    ]

    for i, (col_x, col_y, titulo) in enumerate(pares):
        if col_x in df_clean.columns and col_y in df_clean.columns:
            # Remover zeros para melhor visualização
            mask = (df_clean[col_x] > 0) & (df_clean[col_y] > 0)
            x_data = df_clean.loc[mask, col_x]
            y_data = df_clean.loc[mask, col_y]

            # Scatter plot
            axes[i].scatter(x_data, y_data, alpha=0.5, s=20)

            # Linha de tendência
            if len(x_data) > 1:
                z = np.polyfit(x_data, y_data, 1)
                p = np.poly1d(z)
                axes[i].plot(x_data, p(x_data), 'r--', linewidth=2, label='Linha de Tendência')

                # Calcular correlação
                corr = x_data.corr(y_data)
                axes[i].text(0.05, 0.95, f'r = {corr:.3f}',
                           transform=axes[i].transAxes, fontsize=12,
                           verticalalignment='top', fontweight='bold',
                           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

            axes[i].set_xlabel(f'Número de Consumidores - {titulo}', fontweight='bold')
            axes[i].set_ylabel(f'Energia Faturada (MWh) - {titulo}', fontweight='bold')
            axes[i].set_title(f'Consumidores vs Energia - {titulo}', fontweight='bold')
            axes[i].grid(True, alpha=0.3)
            axes[i].legend()

    plt.tight_layout()
    plt.savefig('scatter_consumidores_energia.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('✓ Scatter plots criados com sucesso!')

## 10. Análise Temporal

Vamos analisar a **evolução temporal** dos dados de subvenções e consumidores.

In [ ]:
# Análise temporal - evolução mensal
if 'AnmMesAnoCompetencia' in df_clean.columns:
    # Agrupar por mês/ano
    df_temporal = df_clean.groupby('AnmMesAnoCompetencia').agg({
        'NumConsBaixaRenda': 'sum',
        'VlrDMR': 'sum',
        'VlrCDE': 'sum'
    }).reset_index()

    # Ordenar por data
    df_temporal = df_temporal.sort_values('AnmMesAnoCompetencia')

    print('=== EVOLUÇÃO TEMPORAL (Primeiras 10 linhas) ===')
    print(df_temporal.head(10))

In [ ]:
# Gráfico de evolução temporal
if 'AnmMesAnoCompetencia' in df_clean.columns and len(df_temporal) > 0:
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))

    # Evolução de consumidores
    axes[0].plot(df_temporal['AnmMesAnoCompetencia'], df_temporal['NumConsBaixaRenda'],
                marker='o', linewidth=2, markersize=4, color='steelblue')
    axes[0].set_ylabel('Total de Consumidores\nBaixa Renda', fontsize=11, fontweight='bold')
    axes[0].set_title('Evolução Temporal - Consumidores Baixa Renda', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)

    # Evolução de DMR
    axes[1].plot(df_temporal['AnmMesAnoCompetencia'], df_temporal['VlrDMR'],
                marker='s', linewidth=2, markersize=4, color='darkorange')
    axes[1].set_ylabel('DMR Total (R$)', fontsize=11, fontweight='bold')
    axes[1].set_title('Evolução Temporal - Diferença Mensal de Receita (DMR)', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)

    # Evolução de CDE
    axes[2].plot(df_temporal['AnmMesAnoCompetencia'], df_temporal['VlrCDE'],
                marker='^', linewidth=2, markersize=4, color='forestgreen')
    axes[2].set_xlabel('Mês/Ano de Competência', fontsize=11, fontweight='bold')
    axes[2].set_ylabel('CDE Total (R$)', fontsize=11, fontweight='bold')
    axes[2].set_title('Evolução Temporal - Conta de Desenvolvimento Energético (CDE)', fontsize=12, fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    axes[2].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig('evolucao_temporal.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('✓ Gráficos de evolução temporal criados!')

## 11. Teste de Normalidade

Como visto na aula, a **distribuição normal** é fundamental em estatística. Vamos testar se nossas variáveis seguem uma distribuição normal usando:
- Visualização (Q-Q plot)
- Teste de Shapiro-Wilk

In [ ]:
# Teste de normalidade para variáveis numéricas
if len(cols_consumidores) > 0:
    print('=== TESTE DE NORMALIDADE (Shapiro-Wilk) ===')
    print('H0: Os dados seguem uma distribuição normal')
    print('H1: Os dados NÃO seguem uma distribuição normal')
    print('Critério: Se p-valor < 0.05, rejeitamos H0 (dados não são normais)\n')

    resultados_normalidade = []

    for col in cols_consumidores[:3]:  # Testar apenas 3 primeiras colunas
        # Remover zeros e pegar amostra (Shapiro-Wilk tem limite de 5000 observações)
        data = df_clean[col][df_clean[col] > 0]
        if len(data) > 5000:
            data = data.sample(5000, random_state=42)

        # Teste de Shapiro-Wilk
        statistic, p_value = stats.shapiro(data)

        resultado = 'Normal' if p_value > 0.05 else 'Não Normal'
        resultados_normalidade.append({
            'Variável': col,
            'Estatística': statistic,
            'p-valor': p_value,
            'Resultado': resultado
        })

    df_normalidade = pd.DataFrame(resultados_normalidade)
    print(df_normalidade.to_string(index=False))

In [ ]:
# Q-Q Plot para avaliar normalidade visualmente
if len(cols_consumidores) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for i, col in enumerate(cols_consumidores[:3]):
        # Remover zeros e pegar amostra
        data = df_clean[col][df_clean[col] > 0]
        if len(data) > 5000:
            data = data.sample(5000, random_state=42)

        # Q-Q plot
        stats.probplot(data, dist='norm', plot=axes[i])
        axes[i].set_title(f'Q-Q Plot\n{col.replace("NumCons", "")}', fontweight='bold')
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('qqplot_normalidade.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('✓ Q-Q Plots criados!')
    print('\n📊 INTERPRETAÇÃO DO Q-Q PLOT:')
    print('• Se os pontos seguem aproximadamente a linha diagonal vermelha, os dados são normais')
    print('• Desvios da linha indicam não-normalidade')
    print('• Caudas pesadas aparecem como curvaturas nas extremidades')

## 12. Análise por Distribuidora (Agente)

Vamos analisar as distribuidoras com maior número de beneficiários e valores de DMR.

In [ ]:
# Top 10 distribuidoras por número de consumidores baixa renda
if 'SigAgente' in df_clean.columns and 'NumConsBaixaRenda' in df_clean.columns:
    top_distribuidoras = df_clean.groupby('SigAgente').agg({
        'NumConsBaixaRenda': 'sum',
        'VlrDMR': 'sum',
        'VlrCDE': 'sum',
        'MdaMWhBaixaRenda': 'sum'
    }).sort_values('NumConsBaixaRenda', ascending=False).head(10)

    print('=== TOP 10 DISTRIBUIDORAS - CONSUMIDORES BAIXA RENDA ===')
    print(top_distribuidoras.round(2))

In [ ]:
# Gráfico das top distribuidoras
if 'SigAgente' in df_clean.columns and 'NumConsBaixaRenda' in df_clean.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Top 10 por número de consumidores
    colors_dist = plt.cm.tab10(np.linspace(0, 1, 10))
    ax1.barh(top_distribuidoras.index, top_distribuidoras['NumConsBaixaRenda'], color=colors_dist, edgecolor='black')
    ax1.set_xlabel('Total de Consumidores Baixa Renda', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Distribuidora (Sigla)', fontsize=11, fontweight='bold')
    ax1.set_title('Top 10 Distribuidoras - Consumidores Baixa Renda', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.invert_yaxis()

    # Top 10 por valor de DMR
    top_dmr = df_clean.groupby('SigAgente')['VlrDMR'].sum().sort_values(ascending=False).head(10)
    ax2.barh(top_dmr.index, top_dmr.values, color=colors_dist, edgecolor='black')
    ax2.set_xlabel('Total DMR (R$)', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Distribuidora (Sigla)', fontsize=11, fontweight='bold')
    ax2.set_title('Top 10 Distribuidoras - Valor de DMR', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    ax2.invert_yaxis()

    plt.tight_layout()
    plt.savefig('top_distribuidoras.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('✓ Gráficos de distribuidoras criados!')

## 13. Paradoxo de Simpson

Como mencionado na aula, o **Paradoxo de Simpson** ocorre quando uma correlação aparente pode ser enganosa ao ignorar variáveis confundidoras (confounding variables).

Vamos investigar se há diferenças nas relações quando analisamos por faixa de consumo.

In [ ]:
# Análise de correlação estratificada por faixa de consumo
if 'IdcFaixa' in df_clean.columns and 'NumConsBaixaRenda' in df_clean.columns and 'MdaMWhBaixaRenda' in df_clean.columns:
    print('=== ANÁLISE DE CORRELAÇÃO ESTRATIFICADA POR FAIXA DE CONSUMO ===')
    print('(Investigando possível Paradoxo de Simpson)\n')

    # Correlação geral
    mask_geral = (df_clean['NumConsBaixaRenda'] > 0) & (df_clean['MdaMWhBaixaRenda'] > 0)
    corr_geral = df_clean.loc[mask_geral, 'NumConsBaixaRenda'].corr(df_clean.loc[mask_geral, 'MdaMWhBaixaRenda'])
    print(f'Correlação GERAL (todas as faixas): r = {corr_geral:.4f}\n')

    # Correlação por faixa
    print('Correlação POR FAIXA DE CONSUMO:')
    for faixa in sorted(df_clean['IdcFaixa'].unique()):
        df_faixa = df_clean[df_clean['IdcFaixa'] == faixa]
        mask_faixa = (df_faixa['NumConsBaixaRenda'] > 0) & (df_faixa['MdaMWhBaixaRenda'] > 0)

        if mask_faixa.sum() > 1:  # Precisa de pelo menos 2 pontos para calcular correlação
            corr_faixa = df_faixa.loc[mask_faixa, 'NumConsBaixaRenda'].corr(df_faixa.loc[mask_faixa, 'MdaMWhBaixaRenda'])
            print(f'  Faixa {int(faixa)}: r = {corr_faixa:.4f} (n = {mask_faixa.sum():,} registros)')

    print('\n📊 Nota: Se as correlações por faixa diferem substancialmente da correlação geral,')
    print('   isso pode indicar um Paradoxo de Simpson!')

## 14. Resumo das Principais Descobertas

Vamos consolidar as principais estatísticas e insights obtidos na análise exploratória.

In [ ]:
# Resumo executivo
print('='*80)
print('RESUMO EXECUTIVO - ANÁLISE EXPLORATÓRIA DE DADOS'.center(80))
print('='*80)

print('\n1️⃣ DIMENSÕES DO DATASET')
print(f'   • Total de registros: {len(df_clean):,}')
print(f'   • Total de variáveis: {len(df_clean.columns)}')
print(f'   • Período analisado: {df_clean["AnmMesAnoCompetencia"].min().strftime("%m/%Y")} a {df_clean["AnmMesAnoCompetencia"].max().strftime("%m/%Y")}')

if len(cols_consumidores) > 0:
    print('\n2️⃣ TENDÊNCIAS CENTRAIS (Consumidores Baixa Renda)')
    print(f'   • Média: {df_clean["NumConsBaixaRenda"].mean():,.2f}')
    print(f'   • Mediana: {df_clean["NumConsBaixaRenda"].median():,.2f}')
    print(f'   • Desvio Padrão: {df_clean["NumConsBaixaRenda"].std():,.2f}')
    print(f'   • Valor Mínimo: {df_clean["NumConsBaixaRenda"].min():,.2f}')
    print(f'   • Valor Máximo: {df_clean["NumConsBaixaRenda"].max():,.2f}')

if 'VlrDMR' in df_clean.columns:
    print('\n3️⃣ VALORES FINANCEIROS (DMR)')
    print(f'   • DMR Total: R$ {df_clean["VlrDMR"].sum():,.2f}')
    print(f'   • DMR Médio: R$ {df_clean["VlrDMR"].mean():,.2f}')
    print(f'   • DMR Mediano: R$ {df_clean["VlrDMR"].median():,.2f}')

if 'IdcFaixa' in df_clean.columns:
    print('\n4️⃣ DISTRIBUIÇÃO POR FAIXA DE CONSUMO')
    moda_faixa = df_clean['IdcFaixa'].mode()[0]
    print(f'   • Faixa mais comum: Faixa {int(moda_faixa)}')
    print(f'   • Número de faixas distintas: {df_clean["IdcFaixa"].nunique()}')

if 'SigAgente' in df_clean.columns:
    print('\n5️⃣ DISTRIBUIDORAS')
    print(f'   • Total de distribuidoras: {df_clean["SigAgente"].nunique()}')
    top_dist = df_clean.groupby('SigAgente')['NumConsBaixaRenda'].sum().idxmax()
    print(f'   • Distribuidora com mais beneficiários: {top_dist}')

print('\n' + '='*80)
print('✓ Análise Exploratória Concluída com Sucesso!'.center(80))
print('='*80)

## 15. Conclusões e Próximos Passos

### Conceitos de EDA Aplicados (conforme Aula 02)

✅ **Descrição de Conjuntos de Dados**
- Análise inicial com `info()`, `describe()`, `head()`
- Identificação de tipos de dados e valores ausentes

✅ **Tendências Centrais**
- Média, mediana e moda calculadas para todas as variáveis
- Comparação entre média e mediana para identificar assimetria
- Quantis (Q1, Q2, Q3) para compreensão da distribuição

✅ **Dispersão**
- Range (intervalo) entre valores mínimos e máximos
- Variância e desvio padrão
- IQR (Intervalo Interquartil) como medida robusta
- Coeficiente de variação para comparar dispersões

✅ **Correlação**
- Matriz de correlação de Pearson
- Heatmap para visualização de correlações
- Scatter plots com linhas de tendência

✅ **Visualizações**
- Histogramas para distribuições
- Box plots para identificar outliers
- Gráficos de linha para séries temporais
- Gráficos de barras e pizza para categorias

✅ **Distribuições**
- Teste de normalidade (Shapiro-Wilk)
- Q-Q plots para avaliação visual
- Identificação de distribuições com cauda pesada

✅ **Paradoxo de Simpson**
- Análise estratificada por faixa de consumo
- Comparação de correlações gerais vs. segmentadas

### Próximos Passos Sugeridos

1. **Modelagem Preditiva**: Desenvolver modelos de previsão de DMR e consumo
2. **Análise de Séries Temporais**: Aplicar técnicas de decomposição e previsão
3. **Segmentação**: Clustering de distribuidoras e consumidores
4. **Análise de Anomalias**: Detectar padrões atípicos e fraudes
5. **Dashboard Interativo**: Criar visualizações dinâmicas com Plotly/Dash

---

**Referências:**
- Aula 02: Estatística e Probabilidade - Prof. Dr. Raphael Gomes
- ANEEL - Dados Abertos: https://dadosabertos.aneel.gov.br
- Dicionário de Dados: SCS - Sistema de Controle de Subvenções e Programas Sociais